In [1]:
import pandas as pd
import glob
import numpy as np

In [2]:
extension = 'csv'
result_files = glob.glob('*.{}'.format(extension))
print(result_files)
print(len(result_files))

['678_visualizing_environmental.csv', '687_sleuth_ex1605.csv', '659_sleuth_ex1714.csv', '561_cpu.csv', '529_pollen.csv', '503_wind.csv', '1029_LEV.csv', 'Nguyen-9.csv', '522_pm10.csv', 'Nguyen-4.csv', '542_pollution.csv', '1027_ESL.csv', 'Nguyen-12.csv', '1028_SWD.csv', '695_chatfield_4.csv', '225_puma8NH.csv', '227_cpu_small.csv', '229_pwLinear.csv', 'Nguyen-13.csv', 'Nguyen-5.csv', '712_chscase_geyser1.csv', '547_no2.csv', 'Nguyen-1.csv', 'Nguyen-2.csv', 'Nguyen-10.csv', '1096_FacultySalaries.csv', '666_rmftsa_ladata.csv', '192_vineyard.csv', '519_vinnie.csv', '527_analcatdata_election2000.csv', '706_sleuth_case1202.csv', '523_analcatdata_neavote.csv', 'Nguyen-8.csv', '560_bodyfat.csv', 'flex-blackbox_results.csv', 'Nguyen-11.csv', '1030_ERA.csv', '485_analcatdata_vehicle.csv', '505_tecator.csv', '556_analcatdata_apnea2.csv', '690_visualizing_galaxy.csv', 'Nguyen-6.csv', '663_rabe_266.csv', 'Nguyen-7.csv', '557_analcatdata_apnea1.csv', 'Nguyen-3.csv', '197_cpu_act.csv', '665_sleuth_c

In [3]:
r2_tests= []
aggregated_results = pd.DataFrame()
# combine all the results files into one dataset
for file in result_files: 
    # skip aggregate results file
    if "_results" in file:
        continue
    # skip Nguyen datasets
    if "Nguyen" in file:
        continue
    # skip Friedman datasets
    if "fri" in file:
        continue
    data = pd.read_csv(file, sep=";", header=0)
    aggregated_results = pd.concat([aggregated_results, data])
    r2_tests.append(data["r2_test"].to_numpy())

# add algorithm name
aggregated_results["algorithm"] = "Flex"

In [4]:
print(aggregated_results[["algorithm", "problem", "r2_test"]])

   algorithm                        problem   r2_test
0       Flex  678_visualizing_environmental -0.215653
1       Flex  678_visualizing_environmental  0.331855
2       Flex  678_visualizing_environmental  0.070802
3       Flex  678_visualizing_environmental  0.187030
4       Flex  678_visualizing_environmental -0.046592
..       ...                            ...       ...
5       Flex                    228_elusage  0.673283
6       Flex                    228_elusage  0.831048
7       Flex                    228_elusage  0.834349
8       Flex                    228_elusage  0.663896
9       Flex                    228_elusage  0.353254

[380 rows x 3 columns]


In [5]:
aggregated_results = aggregated_results.rename(columns={"r2_test": "r2_zero_test", "problem": "dataset"})

aggregated_results.to_csv("flex-blackbox_results.csv", index=False)

In [6]:
# Group by problem and calculate the mean, median, and standard deviation for r2_zero_test scores
algorithm_stats = aggregated_results.groupby("dataset").agg({"r2_train": "median", "r2_zero_test": "median"}).reset_index()

algorithm_stats["r2_difference"] = algorithm_stats["r2_train"] - algorithm_stats["r2_zero_test"]

# Sort algorithms by median r2_zero_test score
algorithm_stats = algorithm_stats.sort_values(by="r2_difference", ascending=False).reset_index(drop=True)

print(algorithm_stats)

                          dataset  r2_train  r2_zero_test  r2_difference
0               687_sleuth_ex1605  0.818541     -0.175287       0.993829
1                    192_vineyard  0.859980      0.235142       0.624838
2                   542_pollution  0.835501      0.232395       0.603106
3         485_analcatdata_vehicle  0.903565      0.305340       0.598225
4               659_sleuth_ex1714  0.948444      0.559240       0.389204
5   678_visualizing_environmental  0.535002      0.181182       0.353820
6                    1089_USCrime  0.949506      0.640551       0.308955
7             665_sleuth_case2002  0.508988      0.205432       0.303556
8             706_sleuth_case1202  0.838933      0.588165       0.250768
9                     228_elusage  0.894745      0.668589       0.226156
10                      210_cloud  0.938448      0.727274       0.211174
11           1096_FacultySalaries  0.971219      0.787317       0.183901
12                       522_pm10  0.359529      0.

In [7]:
# Group by problem and calculate the mean, median, and standard deviation for r2_zero_test scores
algorithm_stats = aggregated_results.groupby("dataset")["r2_zero_test"].agg(['mean', 'median', 'std']).reset_index()

# Sort algorithms by median r2_zero_test score
algorithm_stats = algorithm_stats.sort_values(by="median", ascending=False).reset_index(drop=True)

print(algorithm_stats)

                          dataset      mean    median       std
0    527_analcatdata_election2000  0.995141  0.998633  0.006245
1                    663_rabe_266  0.994959  0.995115  0.001374
2                     560_bodyfat  0.788588  0.990397  0.629368
3                     505_tecator  0.985503  0.985858  0.005670
4                         561_cpu  0.976047  0.979338  0.017228
5          690_visualizing_galaxy  0.961517  0.964482  0.008816
6                     197_cpu_act  0.948710  0.950359  0.011919
7                   227_cpu_small  0.950126  0.949593  0.004849
8         523_analcatdata_neavote  0.936505  0.944117  0.028010
9                 230_machine_cpu  0.861964  0.900816  0.105257
10         557_analcatdata_apnea1  0.875189  0.884001  0.043530
11                       1027_ESL  0.858928  0.863118  0.024530
12         556_analcatdata_apnea2  0.858232  0.851889  0.044331
13                695_chatfield_4  0.853878  0.847723  0.038045
14                   229_pwLinear  0.830

In [8]:
r2_tests = np.concatenate(r2_tests).ravel()
# print(r2_tests)

In [9]:
print("Mean test R2 = ", r2_tests.mean().item())

Mean test R2 =  0.6244201424966784


In [10]:
r2_tests.std().item()

0.4699765998891306

In [11]:
print("Median test R2 = ", np.median(r2_tests).item())

Median test R2 =  0.7455753572648223


In [12]:
# Convert the DataFrame to Markdown
markdown_table = algorithm_stats.to_markdown(index=False)

# Print the Markdown table
print(markdown_table)

# Save the Markdown table to a file
with open('table.md', 'w') as file:
    file.write(markdown_table)


| dataset                       |       mean |    median |        std |
|:------------------------------|-----------:|----------:|-----------:|
| 527_analcatdata_election2000  |  0.995141  |  0.998633 | 0.00624497 |
| 663_rabe_266                  |  0.994959  |  0.995115 | 0.00137354 |
| 560_bodyfat                   |  0.788588  |  0.990397 | 0.629368   |
| 505_tecator                   |  0.985503  |  0.985858 | 0.00567045 |
| 561_cpu                       |  0.976047  |  0.979338 | 0.0172276  |
| 690_visualizing_galaxy        |  0.961517  |  0.964482 | 0.00881618 |
| 197_cpu_act                   |  0.94871   |  0.950359 | 0.0119189  |
| 227_cpu_small                 |  0.950126  |  0.949593 | 0.00484856 |
| 523_analcatdata_neavote       |  0.936505  |  0.944117 | 0.0280097  |
| 230_machine_cpu               |  0.861964  |  0.900816 | 0.105257   |
| 557_analcatdata_apnea1        |  0.875189  |  0.884001 | 0.0435301  |
| 1027_ESL                      |  0.858928  |  0.863118 | 0.024